# airline_series_check_v1
항공 수송통계(kr_airline_traffic) 시리즈가 매달 제대로 이어지는지 점검합니다. 읽기만 하고 DB는 바꾸지 않습니다.


In [1]:
import sys
from pathlib import Path
import pandas as pd

# DATA 폴더를 찾아 config 불러오기 (investment_strategy 아래 어디서 실행해도 됨)
for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / "DATA").exists():
        for p in (str(parent), str(parent / "DATA")):
            if p not in sys.path:
                sys.path.insert(0, p)
        break
import config
engine = config.get_engine(config.get_db_info())

sql = """
SELECT i.source,
       COUNT(DISTINCT i.series_id)                         AS n_series,
       COUNT(d.period_end) / COUNT(DISTINCT i.series_id)   AS avg_obs,
       COUNT(DISTINCT d.period_end)                        AS n_months,
       MIN(d.period_end) AS first_month,
       MAX(d.period_end) AS last_month
FROM kr_airline_traffic_info i
LEFT JOIN kr_airline_traffic d ON d.series_id = i.series_id
GROUP BY i.source
"""
res = pd.read_sql(sql, engine)
pd.set_option("display.width", 200)
print(res.to_string(index=False))
print()

# 자동 판정: 시리즈당 평균 관측 수가 수집 개월 수와 비슷하면 정상
for r in res.itertuples():
    ratio = r.avg_obs / r.n_months if r.n_months else 0
    if ratio >= 0.8:
        verdict = "정상 (시리즈가 매달 이어지고 있음)"
    elif ratio >= 0.5:
        verdict = "주의 (일부 시리즈가 중간에 끊기거나 새로 생김)"
    else:
        verdict = "문제 (같은 시계열이 매달 새 시리즈로 쪼개지는 것으로 보임)"
    print(f"{r.source:<12} 수집 {r.n_months}개월, 시리즈당 평균 {r.avg_obs:.1f}개월 → {verdict}")

     source  n_series  avg_obs  n_months first_month last_month
  AIRPORTAL       488   1.9672         2  2026-07-31 2026-08-31
KAC_AIRPORT       546   1.9744         2  2026-07-31 2026-08-31
ODP_B551177        24   2.0000         2  2026-07-31 2026-08-31

AIRPORTAL    수집 2개월, 시리즈당 평균 2.0개월 → 정상 (시리즈가 매달 이어지고 있음)
KAC_AIRPORT  수집 2개월, 시리즈당 평균 2.0개월 → 정상 (시리즈가 매달 이어지고 있음)
ODP_B551177  수집 2개월, 시리즈당 평균 2.0개월 → 정상 (시리즈가 매달 이어지고 있음)
